<a href="https://colab.research.google.com/github/prakash587/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prakash587/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

## 1. Distributions

I first checked the distributions of the main fields used in the signal audit: `impressions_90d`, `avg_position`, `ctr`, and `days_since_last_update`.

The traffic-related fields are expected to be right-skewed, with a smaller number of pages receiving much more traffic than the typical page. Because of this, I avoid relying on raw Pearson correlations and use grouped buckets for the signal tests instead.

I also checked missingness and the special meaning of `avg_position = 0`. The data dictionary states that this represents no position data rather than a true rank of zero. Therefore, I exclude these rows when testing position-related signals.

The distributions are useful context because large traffic values can otherwise dominate a simple correlation. The following tests therefore use readable buckets and show the sample size (`n`) for each bucket.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Label used only for signal auditing
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("Dataset shape:", df.shape)

# Key fields used in the signal audit
key_fields = [
    "impressions_90d",
    "avg_position",
    "ctr",
    "days_since_last_update"
]

print("\nSummary statistics:")
print(df[key_fields].describe().T)

print("\nSelected percentiles:")
print(
    df[key_fields].quantile(
        [0.25, 0.50, 0.75, 0.90, 0.99]
    ).T
)

print("\nMissing values:")
print(df[key_fields].isna().sum())

print("\nPosition no-data rows:")
print((df["avg_position"] == 0).sum())

print("\nImpressions distribution:")
print(df["impressions_90d"].describe())

print("\nCTR distribution:")
print(df["ctr"].describe())

Dataset shape: (30000, 45)

Summary statistics:
                          count         mean           std  min   25%     50%  \
impressions_90d         30000.0  5200.366300  16838.019547  1.0  81.0  731.00   
avg_position            30000.0    16.342380     15.216790  0.0   6.2   10.80   
ctr                     30000.0     0.510733      3.279162  0.0   0.0    0.07   
days_since_last_update  30000.0    46.098300     42.078709  1.0  20.0   20.00   

                            75%       max  
impressions_90d         3615.25  517715.0  
avg_position              22.30     245.0  
ctr                        0.29     100.0  
days_since_last_update   104.00     373.0  

Selected percentiles:
                        0.25    0.50     0.75      0.90       0.99
impressions_90d         81.0  731.00  3615.25  12136.40  73505.830
avg_position             6.2   10.80    22.30     36.80     69.901
ctr                      0.0    0.07     0.29      0.65      8.330
days_since_last_update  20.0   20.0

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

## 2. Signal tests

### Signal 1 — Freshness vs. decline rate

**Claim:** Older content should be more likely to show a declining trend.

**Test:** I compared the `is_declining_label` rate across the four `freshness_tier` groups and reported the number of rows in each group.

**Observed result:**

| freshness_tier | decline rate |      n |
| -------------- | -----------: | -----: |
| 0-30           |        0.511 | 20,480 |
| 31-90          |        0.589 |    175 |
| 91-180         |        0.611 |  9,171 |
| 181+           |        0.471 |    174 |

**Verdict: MIXED**

The relationship is not monotonic. The 91-180 group has the highest observed decline rate, but the 181+ group has the lowest rate. The two extreme groups also have relatively small sample sizes compared with the main groups. Therefore, the data does not provide a consistent enough staleness pattern to use staleness alone as a strong decline signal.

### Signal 2 — Position vs. CTR

**Claim:** CTR should decrease as search position becomes worse.

**Test:** I compared mean CTR across the position tiers, excluding `no_data`.

**Observed result:**

| position_tier | mean CTR |      n |
| ------------- | -------: | -----: |
| top_3         |    1.484 |  2,321 |
| page_1        |    0.652 | 11,814 |
| striking      |    0.323 |  7,304 |
| page_3_5      |    0.222 |  7,242 |
| deep          |    0.150 |  1,319 |

**Verdict: CONFIRMED**

The observed CTR decreases consistently as position becomes worse, and every bucket has substantially more than the minimum sample-size floor. This supports the assumption behind the CTR-fix logic: CTR should be evaluated relative to the page's position rather than using one global CTR threshold.

### Signal 3 — Search volume vs. decline rate

**Claim:** Pages with more search impressions may provide more valuable opportunities when prioritising content actions.

**Test:** I divided `impressions_90d` into five traffic buckets and compared the declining-label rate and sample size for each bucket.

**Verdict:** See the executed table above. I use the observed bucket pattern rather than assuming that higher traffic causes more decline.

This test is mainly used to understand whether traffic volume is useful for prioritisation. It should not be interpreted as evidence that traffic volume causes content decline.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---------------------------------------------------------
# Signal 1: Freshness vs decline
# ---------------------------------------------------------

signal_1 = (
    df.groupby("freshness_tier")["is_declining_label"]
      .agg(["mean", "count"])
      .rename(columns={
          "mean": "decline_rate",
          "count": "n"
      })
)

print("Signal 1 — Freshness vs decline rate")
print(signal_1)

print("\n")


# ---------------------------------------------------------
# Signal 2: Position vs CTR
# ---------------------------------------------------------

position_order = [
    "top_3",
    "page_1",
    "striking",
    "page_3_5",
    "deep"
]

signal_2 = (
    df[df["position_tier"] != "no_data"]
    .groupby("position_tier")["ctr"]
    .agg(["mean", "count"])
    .rename(columns={
        "mean": "mean_ctr",
        "count": "n"
    })
    .reindex(position_order)
)

print("Signal 2 — Position vs CTR")
print(signal_2)

print("\n")


# ---------------------------------------------------------
# Signal 3: Search volume vs decline
# ---------------------------------------------------------

# Use fixed, readable traffic buckets.
# qcut creates approximately equal-sized groups.
df["impressions_bucket"] = pd.qcut(
    df["impressions_90d"],
    q=5,
    duplicates="drop"
)

signal_3 = (
    df.groupby("impressions_bucket", observed=True)["is_declining_label"]
      .agg(["mean", "count"])
      .rename(columns={
          "mean": "decline_rate",
          "count": "n"
      })
)

print("Signal 3 — Search volume vs decline rate")
print(signal_3)

Signal 1 — Freshness vs decline rate
                decline_rate      n
freshness_tier                     
0-30                0.511377  20480
181+                0.471264    174
31-90               0.588571    175
91-180              0.611057   9171


Signal 2 — Position vs CTR
               mean_ctr      n
position_tier                 
top_3          1.483611   2321
page_1         0.652467  11814
striking       0.323239   7304
page_3_5       0.222484   7242
deep           0.150212   1319


Signal 3 — Search volume vs decline rate
                    decline_rate     n
impressions_bucket                    
(0.999, 39.0]           0.325112  6041
(39.0, 364.0]           0.603119  5964
(364.0, 1375.0]         0.605303  5997
(1375.0, 5167.6]        0.633211  5998
(5167.6, 517715.0]      0.545500  6000


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

## 3. The flag-linked test

The baseline rule relies on the CTR-fix assumption that a page should be compared with other pages at a similar search position.

**Claim:** A page's CTR should be judged relative to its position because average CTR changes substantially across search-position tiers.

**Test:** I calculated both mean and median CTR for each `position_tier`, excluding rows where position is `no_data`. I also checked whether the mean CTR decreases monotonically as position gets worse.

**Observed result:** Mean CTR falls from 1.484 for `top_3` pages to 0.150 for `deep` pages, with large sample sizes in every tier. The decrease is monotonic across the five position groups.

**Verdict: CONFIRMED**

This supports the assumption behind the CTR-fix flag. A single global CTR threshold would mix pages with very different expected CTRs because position has a strong relationship with observed CTR. Using a position-tier benchmark is therefore more defensible as a decision-support rule.

This does not prove that a low CTR is caused by a weak title or metadata. It only shows that position must be considered when identifying unusually low CTR.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Flag-linked test:
# Does CTR remain meaningfully different across search-position tiers?

flag_linked = (
    df[df["position_tier"] != "no_data"]
    .groupby("position_tier")
    .agg(
        mean_ctr=("ctr", "mean"),
        median_ctr=("ctr", "median"),
        n=("ctr", "size")
    )
    .reindex([
        "top_3",
        "page_1",
        "striking",
        "page_3_5",
        "deep"
    ])
)

print("Flag-linked test — CTR by position tier")
print(flag_linked)

# Simple monotonicity check
ctr_values = flag_linked["mean_ctr"].dropna().values

is_monotonic = all(
    ctr_values[i] >= ctr_values[i + 1]
    for i in range(len(ctr_values) - 1)
)

print("\nMean CTR decreases monotonically with worse position:",
      is_monotonic)

print("\nSmallest bucket n:",
      int(flag_linked["n"].min()))

Flag-linked test — CTR by position tier
               mean_ctr  median_ctr      n
position_tier                             
top_3          1.483611        0.00   2321
page_1         0.652467        0.16  11814
striking       0.323239        0.11   7304
page_3_5       0.222484        0.03   7242
deep           0.150212        0.00   1319

Mean CTR decreases monotonically with worse position: True

Smallest bucket n: 1319


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

The audit suggests that staleness alone is not a reliable way to identify declining pages because the observed decline rate is mixed across freshness tiers. In contrast, the position-versus-CTR relationship is strong and consistent, so a content team can use position-adjusted CTR as a decision-support signal when prioritising pages for CTR review. Traffic volume can then help prioritise which opportunities are worth investigating first, but it should not be treated as evidence that a page is declining.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.